In [ ]:
#--------Neural Algorithmic Reasoning on Graphs--Instead of explicitly programming every step of an algorithm, train a neural network to learn patterns that resemble those steps.
"""
Suppose a patient history contains these events:
Diagnosis → Biopsy → Biomarker_Test → Treatment → Response / CT scan

The model receives a starting event, the current event, and a target event. It learns to predict the next node on a shortest 
path toward the target.The training examples were generated by a conventional shortest-path algorithm. The neural network 
learned to imitate the next-step decisions from those examples.
"""
import random
from collections import deque
import torch
from torch import nn
random.seed(42)
torch.manual_seed(42)
#----------Block 1: Create the oncology graph
nodes = [                                       # Directed oncology history graph.
    "Diagnosis",
    "Biopsy",
    "Biomarker_Test",
    "Treatment",
    "Response",
    "CT_Scan"
]
graph = {
    "Diagnosis"         : ["Biopsy"],
    "Biopsy"            : ["Biomarker_Test"],
    "Biomarker_Test"    : ["Treatment"],
    "Treatment"         : ["Response", "CT_Scan"],
    "Response"          : [],
    "CT_Scan"           : []
}
#---------End of Block 1: Create the oncology graph

#-----Block 4: Convert events into numbers and build the neural network
node_id = {name: i for i, name in enumerate(nodes)}
# Neural networks work with numerical data, so the event names must be converted into numbers.
# -- So Diagnosis becomes 0, Biopsy 1, Biomarker_Test 2 ,Treatment 3,Response 4, CT_Scan 5
#--- End of Block 4: Convert events into numbers and build the neural network


#--------Block 2: Find the shortest path using BFS----
def shortest_path(start, target):               # Generate examples using a known shortest-path algorithm.
    queue = deque([(start, [start])])           # Stores events waiting to be explored
    visited = {start}                           # Tracks events already visited
    while queue:
        current, path = queue.popleft()         # Takes the next event waiting in the queue
        if current == target:                   # Checks whether the destination is reached
            return path                         # Returns the discovered path
        for neighbor in graph[current]:         # Gets the current event's outgoing connections
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, path + [neighbor]))
    return None

"""
Suppose the question is:
“Starting at Diagnosis, how do I reach Response?”
The result is:
Diagnosis → Biopsy → Biomarker_Test → Treatment → Response / CT scan
The function returns the complete path. It does not use AI to find this path; it uses a conventional algorithm.
"""
#--------End of Block 2: Find the shortest path using BFS
#-------Block 3: Generate training examples
# It creates the examples the AI will learn from.. X (input data) and y (label or target data)
X, y = [], []                    # X stores the questions given to the AI. y stores the correct answers the AI should learn to predict.
for start in nodes:
    for target in nodes:
        path = shortest_path(start, target)
        if path is None or len(path) < 2:
            continue
        for i in range(len(path) - 1):
            current = path[i]
            next_node = path[i + 1]
#---------End of-------Block 3: Generate training examples
#---------Below is continuation of block 4 -- one-hot encoding:
            features = [0] * (3 * len(nodes))
            features[node_id[start]] = 1
            features[len(nodes) + node_id[current]] = 1
            features[2 * len(nodes) + node_id[target]] = 1
"""
The AI receives three pieces of information:
1. Where did the journey start?
2. Where are we currently?
3. Where do we want to go?
Each piece gets six numerical positions, one for each possible event. Therefore, the input contains (3 times 6 = 18) numbers.
For the question Diagnosis → ? → Response, the input is represented as follows:

Start event — Diagnosis 
1 0 0 0 0 0 
Current event — Diagnosis
1 0 0 0 0 0 
Target event — Response
0 0 0 0 1 0 
Each row represents six event positions. Green 1 means the corresponding event is selected; 0 means it is not selected.
"""

#-----End of continuation of block 4            
            X.append(features)
            y.append(node_id[next_node])
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)
model = nn.Sequential(
    nn.Linear(3 * len(nodes), 32),              # takes 18 inputs and calculates 32 intermediate values.The number 32 is a design choice made by the programmer.
    nn.ReLU(),                                  # applies a simple nonlinear transformation, helping the network learn patterns beyond a purely linear relationship.
    nn.Linear(32, len(nodes))                   # produces six scores, one for each possible next event.
)
"""
    18 input numbers
    Start + Current + Target

    32 hidden neurons
    Learn numerical patterns using weights and ReLU

    6 output scores
    One score for each possible next event

    Six output scores do not mean six valid next steps. They mean the model considers six possible node labels, 
    and your code chooses the highest-scoring one.

"""
#-----Block 5: Train the model - the network is learning to imitate BFS on this small, fixed graph. 
# It is not independently discovering the shortest-path algorithm or learning oncology knowledge.
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
loss_function = nn.CrossEntropyLoss()
for epoch in range(500):                          # Repeat this process 500 times
    predictions = model(X)                        # Ask the network to predict the answers
    loss = loss_function(predictions, y)          # Measure how wrong the predictions are
    optimizer.zero_grad()                         # Clear gradients from the previous training step
    loss.backward()                               # Calculate how the weights contributed to the error
    optimizer.step()                              # Update the weights to improve predictions
#----------End of Block 5: Train the model

#------Block 6: Ask the trained model to predict
def predict_next(start, current, target):          # Ask the trained model to navigate from Diagnosis to Response.
    features = [0] * (3 * len(nodes))
    features[node_id[start]] = 1
    features[len(nodes) + node_id[current]] = 1
    features[2 * len(nodes) + node_id[target]] = 1
    with torch.no_grad():
        output = model(torch.tensor([features], dtype=torch.float32))
        next_id = output.argmax(dim=1).item()        # argmax() selects the highest score. It does not count how many possible next events exist.
    return nodes[next_id]

#-------End of Block 6: Ask the trained model to predict
print("Learned next event:")
print(
    "From Diagnosis toward Response:",
    predict_next("Diagnosis", "Diagnosis", "Response")
)

Learned next event:
From Diagnosis toward Response: Biopsy
